# SmartLearn Data Analysis Notebook

Phân tích dữ liệu từ database của ứng dụng SmartLearn

In [ ]:
import sqlite3
import pandas as pd
import os

DB_PATH = 'src/backend/database.db'

def get_connection():
    return sqlite3.connect(DB_PATH)

## 1. Load dữ liệu Users

In [ ]:
conn = get_connection()
users_df = pd.read_sql_query('SELECT * FROM users', conn)
print(f"Tổng số users: {len(users_df)}")
users_df.head()

## 2. Load dữ liệu History (kết quả bài làm)

In [ ]:
history_df = pd.read_sql_query('SELECT * FROM history', conn)
print(f"Tổng số bài làm: {len(history_df)}")
history_df.head()

## 3. Load dữ liệu Questions

In [ ]:
questions_df = pd.read_sql_query('SELECT * FROM questions', conn)
print(f"Tổng số câu hỏi: {len(questions_df)}")
print(f"\nSố câu hỏi theo môn:")
print(questions_df['subject'].value_counts())

## 4. Thống kê điểm số theo môn

In [ ]:
score_stats = history_df.groupby('subject')['score'].agg(['mean', 'min', 'max', 'count'])
score_stats.columns = ['Điểm TB', 'Điểm thấp nhất', 'Điểm cao nhất', 'Số lần làm bài']
score_stats.round(2)

## 5. Top users có điểm cao nhất

In [ ]:
user_scores = history_df.groupby('userId')['score'].mean().sort_values(ascending=False)
user_scores = user_scores.reset_index()
user_scores = user_scores.merge(users_df[['id', 'fullname', 'email']], left_on='userId', right_on='id')
user_scores = user_scores[['fullname', 'email', 'score']].rename(columns={'score': 'Điểm TB'})
user_scores.head(10)

## 6. Tỉ lệ đạt/m không đạt (>=5 điểm)

In [ ]:
history_df['passed'] = history_df['score'] >= 5
pass_rate = history_df['passed'].value_counts(normalize=True) * 100
print(f"Tỉ lệ đạt (>=5 điểm): {pass_rate.get(True, 0):.1f}%")
print(f"Tỉ lệ không đạt: {pass_rate.get(False, 0):.1f}%")

## 7. Đóng kết nối

In [ ]:
conn.close()